In [1]:
# Install necessary packages
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

  Using cached opentelemetry_api-1.33.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached opentelemetry_sdk-1.33.1-py3-none-any.whl.metadata (1.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 233.4 kB/s eta 0:00:00a 0:00:01
  Using cached beautifulsoup4-4.13.4-py3-none-any.whl.metadata (3.8 kB)
  Using cached pytest-8.3.5-py3-none-any.whl.metadata (7.6 kB)
  Using cached fastapi-0.115.12-py3-none-any.whl.metadata (27 kB)
  Using cached uvicorn-0.34.2-py3-none-any.whl.metadata (6.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 661.0 kB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 366.0 kB/s eta 0:00:0000:0100:01
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at m

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Core imports
import os
from crewai import Agent, Task, Crew
from utils import get_openai_api_key, get_serper_api_key

In [5]:
# Set up environment
openai_api_key = get_openai_api_key()
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [6]:
# Tool imports
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

In [7]:
# Initialize tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='./fake_resume.md')
semantic_search_resume = MDXSearchTool(mdx='./fake_resume.md')

Inserting batches in chromadb: 100%|██████████| 1/1 [00:01<00:00,  1.98s/it]


In [8]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on "
         "job posting to help job applicants",
    tools = [scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in "
        "navigating and extracting critical "
        "information from job postings is unmatched."
        "Your skills help pinpoint the necessary "
        "qualifications and skills sought "
        "by employers, forming the foundation for "
        "effective application tailoring."
    )
)

In [9]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

In [10]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a "
         "resume stand out in the job market.",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    )
)

In [11]:
# Agent 4: Interview Preparer
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points "
         "based on the resume and job requirements",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of "
        "interviews. With your ability to formulate key questions "
        "and talking points, you prepare candidates for success, "
        "ensuring they can confidently address all aspects of the "
        "job they are applying for."
    )
)

In [12]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "Analyze the job posting URL provided ({job_posting_url}) "
        "to extract key skills, experiences, and qualifications "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True
)

In [13]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs, and personal write-up "
        "({personal_writeup}). Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)

In [14]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "Using the profile and job requirements obtained from "
        "previous tasks, tailor the resume to highlight the most "
        "relevant areas. Employ tools to adjust and enhance the "
        "resume content. Make sure this is the best resume even but "
        "don't make up any information. Update every section, "
        "inlcuding the initial summary, work experience, skills, "
        "and education. All to better reflrect the candidates "
        "abilities and how it matches the job posting."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    output_file="tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [15]:
# Task for Interview Preparer Agent: Develop Interview Materials
interview_preparation_task = Task(
    description=(
        "Create a set of potential interview questions and talking "
        "points based on the tailored resume and job requirements. "
        "Utilize tools to generate relevant questions and discussion "
        "points. Make sure to use these question and talking points to "
        "help the candiadte highlight the main points of the resume "
        "and how it matches the job posting."
    ),
    expected_output=(
        "A document containing key questions and talking points "
        "that the candidate should prepare for the initial interview."
    ),
    output_file="interview_materials.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)


In [16]:
# Assemble the crew
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

In [17]:
job_application_inputs = {
    'job_posting_url': 'https://careers.deepmind.com/jobs/ai-research-engineer',  # Example AI/ML job posting
    'github_url': 'https://github.com/justthzz',
    'personal_writeup': """Thanuja Liyanage is a passionate and driven AI/ML Engineer in training, currently studying at the University of Westminster. 
    She has hands-on experience in deep learning, NLP, and LLMs, and has built several projects demonstrating practical applications of AI. 
    Thanuja’s background blends academic rigor with a creative approach to problem-solving. Her GitHub showcases projects ranging from 
    LLM agents using CrewAI to RAG systems and NLP pipelines. Thanuja is eager to bring her technical skills and curiosity to a team 
    focused on impactful AI innovation."""
}

In [18]:
# Run the crew
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the job posting URL provided (https://careers.deepmind.com/jobs/ai-research-engineer) to extract key skills, experiences, and qualifications required. Use the tools to gather content and identify and categorize the requirements.
 [DEBUG]: == [Tech Job Researcher] Task output: 


 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://github.com/justthzz) URLs, and personal write-up (Thanuja Liyanage is a passionate and driven AI/ML Engineer in training, currently studying at the University of Westminster. 
    She has hands-on experience in deep learning, NLP, and LLMs, and has built several projects demonstrating practical applications of AI. 
    Thanuja’s background blends academic rigor with a creative approach to problem-solving. Her GitHub showcases projects ranging from 
    LLM agents usi

In [19]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

# Thanuja Liyanage

**AI/ML Engineer**

---

## 🧠 Summary  
Passionate and driven AI/ML Engineer with a focus on leveraging cutting-edge technologies like Large Language Models (LLMs) and Natural Language Processing (NLP) to build intelligent systems. Currently exploring research and applications in Retrieval-Augmented Generation (RAG), autonomous agents, and graph-based workflows.

---

## 🧰 Skills  
- AI/ML Expertise
- Deep Learning
- Natural Language Processing (NLP)
- Large Language Models (LLMs)
- Full Stack Development
- Python
- Machine Learning
- Algorithm Design
- Communication Skills

---

## 🎓 Education  
**University of Westminster**  
B.Sc. in Computer Science – Specializing in AI/ML

---

## 💼 Project Experiences  
- LLM agents using CrewAI
- RAG systems
- NLP pipelines
- Autonomous multi-agent pipelines using CrewAI
- RAG-based QA systems integrating LangGraph
- Multi-turn dialogue agents via AutoGen

---

## 🚀 Interests  
- AI/ML Innovation
- Problem-solving
- Full Stack Development
- Applied Artificial Intelligence
- Generative AI & Large Language Models
- Knowledge Graphs & Autonomous Systems

---

## 📱 Contact Information  
- Instagram: just.thzz
- LinkedIn: [LinkedIn Profile](https://www.linkedin.com/in/justthzz)
- GitHub: [justthzz GitHub Profile](https://github.com/justthzz)

---

## 🏆 Achievements  
- Achievementsx3

---

## 🌍 Location  
Colombo, Sri Lanka

---

## 💻 Tech Stack  
- Full Stack Tools

---

## 🎯 Personal Statement  
"A script kiddie turning AI into reality."

---

## 🛠️ Tech Job Researcher  
*Note: Please research and include any additional relevant skills or experiences required for the AI Research Engineer position at Google DeepMind India.*

---

## 🤖 Engineering Interview Preparer  
*Note: Please prepare Thanuja for potential interview questions related to AI/ML, algorithm design, and large scale system engineering.*

---

In [20]:
display(Markdown("./interview_materials.md"))

Based on Thanuja's resume and the job requirements for the AI Research Engineer position at Google DeepMind India, here are some potential interview questions and talking points for her:

1. Can you explain your experience with building autonomous multi-agent pipelines using CrewAI?
2. How have you applied Large Language Models (LLMs) in your projects, especially in developing RAG-based QA systems integrating LangGraph?
3. What challenges have you faced when working on conversational AI projects, like experimenting with multi-turn dialogue agents via Autogen?
4. How do you approach problem-solving in the realm of AI/ML innovation, and can you provide an example of a particularly challenging problem you've solved?
5. Can you walk us through a project where you implemented Deep Learning techniques and the impact it had on the final product?
6. How do you keep yourself updated with the latest advancements in AI technologies and frameworks like JAX?
7. How do you ensure effective communication and collaboration within a multi-stakeholder environment, especially when working on large-scale systems?
8. What excites you the most about working on impactful AI innovation, and how do you see yourself contributing to projects at Google DeepMind India?
9. Can you discuss a time when you had to productionize a large-scale system from proof-of-concept through implementation, and the key takeaways from that experience?
10. How do you approach algorithm design when tackling complex AI problems, and can you share an example of a novel algorithm you've developed?

These questions and talking points aim to help Thanuja highlight her technical skills, project experiences, problem-solving abilities, and passion for AI/ML innovation during the interview process.